# Scenario 16: Cost Tracking — Custom LLM Integration as Judge

Second of three cost-tracking demos split out from `14_cost_tracking.ipynb`.

**Focus:** native Galileo auto-instrumentation **plus** a `custom_LLM_boolean_metric` whose **judge** is a custom-integrated LLM (e.g., the Citizens-configured `md0033_openai_gpt4ominiptu` proxied through their LLM Gateway).

## How this differs from notebook 15

| Concern | Notebook 15 | Notebook 16 (this) |
|---|---|---|
| User-trace LLM | `gpt-4o-mini` (direct) | `gpt-4o-mini` (direct) |
| Scorer | `GalileoMetrics.correctness` (built-in, Galileo-managed judge) | `custom_LLM_boolean_metric` (custom prompt) |
| **Judge LLM** | Galileo's default OpenAI judge | **`md0033_openai_gpt4ominiptu` — your custom integration** |
| Cost surfaced | `cost` + `correctness_metric_cost` | `cost` + `custom_LLM_boolean_metric_*_cost` |

## Prerequisite

The org must already have a custom LLM integration configured for the judge model name in `CUSTOM_JUDGE_MODEL` below. Step 3 verifies the model is priced (override row exists) before we use it as a judge — if it isn't, change `CUSTOM_JUDGE_MODEL` to one that is.

## Step 0: Load environment variables

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

GALILEO_SDK_SOURCE = None
for root in [Path.cwd(), *Path.cwd().parents]:
    local_sdk_src = root / 'galileo-python' / 'src'
    if (local_sdk_src / 'galileo').exists():
        GALILEO_SDK_SOURCE = local_sdk_src
        if str(local_sdk_src) not in sys.path:
            sys.path.insert(0, str(local_sdk_src))
        break

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/demos/galileo-test/.env


## Step 0b: Imports and constants

`CUSTOM_JUDGE_MODEL` is the **custom-integrated** model the judge will call. Switch this to whichever model your org has configured.

In [2]:
import json
import time
from urllib.parse import urljoin, quote

from galileo import GalileoMetrics, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.metrics import create_custom_llm_metric
from galileo.openai import openai
from galileo.projects import delete_project
from galileo.resources.api.trace import query_traces_projects_project_id_traces_search_post
from galileo.resources.models.log_records_query_request import LogRecordsQueryRequest
from galileo.resources.models.output_type_enum import OutputTypeEnum
from galileo.scorers import Scorers
from galileo_core.constants.request_method import RequestMethod
from galileo_core.schemas.logging.step import StepType

PROJECT_NAME = 'GalileoEval_S16_CostCustomJudge'
LOG_STREAM_NAME = 'cost-custom-judge'
MODEL = 'gpt-4o-mini'                    # user-trace LLM
CUSTOM_JUDGE_MODEL = 'md0033_openai_gpt4ominiptu'  # custom-integrated judge LLM
CUSTOM_METRIC_NAME = 'custom_LLM_boolean_metric_s16'

print(f'Galileo SDK source: {GALILEO_SDK_SOURCE or "installed package"}')
PROJECT_NAME, LOG_STREAM_NAME, MODEL, CUSTOM_JUDGE_MODEL, CUSTOM_METRIC_NAME

/Users/binliu/Documents/rungalileo/demos/galileo-test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Galileo SDK source: /Users/binliu/Documents/rungalileo/galileo-python/src


('GalileoEval_S16_CostCustomJudge',
 'cost-custom-judge',
 'gpt-4o-mini',
 'md0033_openai_gpt4ominiptu',
 'custom_LLM_boolean_metric_s16')

## Step 1: Initialize Galileo and capture URLs

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

ls = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME) or \
     create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger_inst = galileo_context.get_logger_instance()
project_id = getattr(logger_inst, 'project_id', None)
log_stream_id = getattr(logger_inst, 'log_stream_id', None)

console_base = str(config.console_url)
project_url = urljoin(console_base, f'project/{project_id}')
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
trends_url = f'{log_stream_url}?tab=trends'

print('Log stream :', log_stream_url)
print('Trends tab :', trends_url)

Log stream : https://console-bin-citizens.gcp-dev.galileo.ai/project/af0c35bd-a7b1-4bef-ad32-120a20cec15d/log-streams/8f80737e-68aa-494d-a910-7c1976180b74
Trends tab : https://console-bin-citizens.gcp-dev.galileo.ai/project/af0c35bd-a7b1-4bef-ad32-120a20cec15d/log-streams/8f80737e-68aa-494d-a910-7c1976180b74?tab=trends


## Step 2: Wrapped OpenAI client (auto-instrumented)

User traces still go to gpt-4o-mini. Only the **judge** uses the custom integration.

In [4]:
client = openai.OpenAI()
client

## Step 3 — Verify the custom-integrated judge model is priced

A custom LLM integration only contributes to `metric_cost` if the org has a **price override** for its model name. We check for that override here — if `override` is missing, the judge calls will run but their cost will come back as `None` (per the §1.5 fallback rule).

In [5]:
judge_prices = config.api_client.request(
    method=RequestMethod.GET,
    path=f'/models/{quote(CUSTOM_JUDGE_MODEL, safe="")}/prices',
)
print(f'--- Prices for {CUSTOM_JUDGE_MODEL} ---')
print(json.dumps(judge_prices, indent=2, default=str))

if not judge_prices.get('override'):
    print(
        f'\n[WARN] No price override for {CUSTOM_JUDGE_MODEL}. The judge will run but '
        f'its metric_cost will be None. Configure an override in Settings -> Model Pricing, '
        f'or change CUSTOM_JUDGE_MODEL to a priced model.'
    )
else:
    print(f'\nOK -- override present, judge cost will be priced.')

--- Prices for md0033_openai_gpt4ominiptu ---
{
  "override": {
    "input_price": 1000000.0,
    "output_price": 10000000.0
  }
}

OK -- override present, judge cost will be priced.


## Step 3b — §1.7: Synthesize `num_reasoning_tokens` via the native SDK extractor

`gpt-4o-mini` is not a reasoning model, so OpenAI's response carries no `reasoning_tokens` field. To prove that `num_reasoning_tokens` is fully controllable from the SDK side (not gated on model identity), we monkey-patch `galileo.openai.extractors._parse_usage` to inject a synthetic value derived from `output_tokens`.

This exercises the documented native-SDK path:
- `galileo-python/.../openai/extractors.py:438` (`_parse_usage` — flattens usage dict)
- `galileo-python/.../openai/__init__.py:248` (`span.metrics.num_reasoning_tokens = usage.get("reasoning_tokens", 0) if usage else 0`)
- Schema escape hatch: `LlmMetrics` (`core/.../span.py:90`) declares `model_config = ConfigDict(extra="allow")` — that's the only reason the field can be attached at all.

Step 6 will read `num_reasoning_tokens` back from each trace and confirm the synthetic value propagated. The patch must be applied **before** Step 4 sends traces.

In [6]:
from galileo.openai import extractors as _gx

# Reuse a deterministic synthetic formula: half of output tokens + 7.
# Using output_tokens (not completion_tokens) because _parse_usage has already
# renamed completion_tokens -> output_tokens by the time it returns.
def _synthesize_reasoning_tokens(out_t: int) -> int:
    return (out_t // 2) + 7

# Idempotent install: if we've already wrapped, skip — re-running this cell shouldn't double-wrap.
if not getattr(_gx._parse_usage, '_galileo_synth_reasoning', False):
    _orig_parse = _gx._parse_usage

    def _parse_usage_with_synthetic_reasoning(usage):
        parsed = _orig_parse(usage)
        if parsed and 'reasoning_tokens' not in parsed:
            parsed['reasoning_tokens'] = _synthesize_reasoning_tokens(parsed.get('output_tokens', 0) or 0)
        return parsed

    _parse_usage_with_synthetic_reasoning._galileo_synth_reasoning = True
    _gx._parse_usage = _parse_usage_with_synthetic_reasoning
    print('Installed synthetic reasoning_tokens patch on galileo.openai.extractors._parse_usage.')
else:
    print('Synthetic reasoning_tokens patch already installed; skipping.')

Installed synthetic reasoning_tokens patch on galileo.openai.extractors._parse_usage.


## Step 4: Send native traces, capture token counts

In [7]:
def _usage_in_out(usage):
    if usage is None:
        return 0, 0
    in_t = getattr(usage, 'input_tokens', None) or getattr(usage, 'prompt_tokens', None) or 0
    out_t = getattr(usage, 'output_tokens', None) or getattr(usage, 'completion_tokens', None) or 0
    return in_t, out_t

prompts = [
    'How do I reset my password?',
    'What are your business hours?',
    'Can I get a refund on my last order?',
    'Where is my package right now?',
    'How do I contact a human agent?',
    'Is there a mobile app I can download?',
]

galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

native_usage = []
for prompt in prompts:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a concise customer support agent. Answer in 1-2 sentences.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=80,
    )
    in_t, out_t = _usage_in_out(response.usage)
    native_usage.append({'prompt': prompt, 'in_tokens': in_t, 'out_tokens': out_t})

galileo_context.flush()

for row in native_usage:
    print(f"{row['in_tokens']:>4} in / {row['out_tokens']:>4} out  |  {row['prompt']}")
print(f'\n-> {len(native_usage)} native traces sent.')

  34 in /   34 out  |  How do I reset my password?
  33 in /   16 out  |  What are your business hours?
  37 in /   21 out  |  Can I get a refund on my last order?
  34 in /   18 out  |  Where is my package right now?
  35 in /   33 out  |  How do I contact a human agent?
  36 in /   31 out  |  Is there a mobile app I can download?

-> 6 native traces sent.


## Step 5: Create + enable `custom_LLM_boolean_metric` using the custom-integrated judge

The key parameter is **`model_name=CUSTOM_JUDGE_MODEL`** — that's what tells the runners to call your custom integration when the scorer fires. Each scoring run produces one judge LLM call per `num_judges` per record, and each call's spend lands in `{metric_name}_metric_cost` on the trace metrics.

`enable_metrics()` is per-stream and replaces the active set.

In [ ]:
existing = Scorers().list(name=CUSTOM_METRIC_NAME)
if existing:
    print(f'Reusing metric: {CUSTOM_METRIC_NAME} (scorer_id={existing[0].id})')
else:
    create_custom_llm_metric(
        name=CUSTOM_METRIC_NAME,
        user_prompt=(
            'You are evaluating a customer support chatbot reply.\n'
            'The reply is GOOD if BOTH:\n'
            '  (a) It is at most 2 sentences long.\n'
            '  (b) It does not fabricate specifics (no made-up order IDs, prices, or links).\n'
            'Return true if the reply is GOOD, false otherwise.'
        ),
        node_level=StepType.llm,
        output_type=OutputTypeEnum.BOOLEAN,
        cot_enabled=True,
        model_name=CUSTOM_JUDGE_MODEL,    # <-- routes the judge through the custom integration
        num_judges=1,
        description=f'Boolean LLM-as-judge running on {CUSTOM_JUDGE_MODEL}',
        tags=['demo', 'cost-tracking', 'custom-judge'],
    )
    print(f'Created metric: {CUSTOM_METRIC_NAME} (judge model = {CUSTOM_JUDGE_MODEL})')

enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[CUSTOM_METRIC_NAME],
)
print(f'Metrics enabled on {LOG_STREAM_NAME}: [{CUSTOM_METRIC_NAME}]')

Reusing metric: custom_LLM_boolean_metric_s16 (scorer_id=57cdb48a-2184-49c1-8762-4ca489ff0306)


Metrics enabled on cost-custom-judge: [custom_LLM_boolean_metric_s16]


## Step 6: Verify both cost streams

- `cost` — your gpt-4o-mini user-trace spend (priced from `costs.json` or per-org override).
- `{custom_metric}_metric_cost` — the **custom-integrated judge** spend.

**State machine for metric_cost** (key §2.2 nuance): the trace search response carries each scorer's metric_cost as `MetricSuccess.cost` (`Union[None, Unset, float]`). Three observable states:

| State | What it means |
|---|---|
| `populated` (number) | Judge ran, runners priced its tokens. |
| `null` | Judge ran but cost wasn't reported (e.g. response missing token usage, or no priced model). |
| `absent` | Scorer key not present in `metric_info` yet — Celery scoring job hasn't completed. |

**Important asymmetry uncovered by running this notebook:** even with the override correctly configured in Step 3 (`{"input_price": 0.1, "output_price": 1.0}`) and the judge clearly running (`judge_tok ≈ 800` tokens per call below), `metric_cost` comes back as **`$0.000000`**. The state is `populated`, not `null` — the judge produced a number, but that number is exactly zero.

**Root cause** (verified against `orbit/libs/python/promptgalileo/promptgalileo/schemas/model.py:155-208`): the scorer's per-call cost goes through `Model.cost()`, which uses `self.input_price` / `self.output_price` and falls back only to `CostTable` (`costs.json`). It **does not consult the per-org `model_price_overrides` table** — that table is only read by the trace user-cost path (`runners/utils/costs/cost.py:39-51`).

**Workarounds** to get non-zero `metric_cost` for a custom-integrated judge:
1. Add the judge model name to `costs.json` (PR against orbit).
2. Set non-zero `input_price` / `output_price` directly on the Model schema for the integration.
3. Use one of Galileo's known-priced models as the judge (e.g. `gpt-4o-mini`, which costs.json prices) instead of `md0033_openai_gpt4ominiptu`.

This is the demo's main finding for §2.2.

In [9]:
GPT_4O_MINI_IN_PER_M = 0.15
GPT_4O_MINI_OUT_PER_M = 0.60

def fetch_traces(target_log_stream_id, limit=50):
    req = LogRecordsQueryRequest(log_stream_id=target_log_stream_id, limit=limit)
    return query_traces_projects_project_id_traces_search_post.sync(
        project_id=project_id, client=config.api_client, body=req,
    )

def _props(obj, attr):
    target = getattr(obj, attr, None)
    if target is None or not hasattr(target, 'additional_properties'):
        return {}
    return target.additional_properties or {}

def extract_user_cost(record):
    info = _props(record, 'metric_info')
    cost_metric = info.get('cost')
    return getattr(cost_metric, 'value', None) if cost_metric else None

def extract_scorer_metric_cost(record, scorer_name):
    """Return (state, value, total_tokens). state in {'populated','null','absent'}."""
    info = _props(record, 'metric_info')
    entry = info.get(scorer_name)
    if entry is None:
        return 'absent', None, None
    cost = getattr(entry, 'cost', None)
    total_tokens = getattr(entry, 'total_tokens', None)
    if isinstance(cost, (int, float)):
        return 'populated', float(cost), total_tokens
    return 'null', None, total_tokens

def extract_tokens(record):
    """Trace-level metrics roll up num_input/output/total_tokens. We read from the
    SDK's `additional_properties` since the typed fields aren't populated on the
    trace-search response (only spans carry them as typed)."""
    metrics_obj = getattr(record, 'metrics', None)
    if metrics_obj is None:
        return 0, 0
    ap = _props(record, 'metrics')
    in_t = ap.get('num_input_tokens') or getattr(metrics_obj, 'num_input_tokens', None) or 0
    out_t = ap.get('num_output_tokens') or getattr(metrics_obj, 'num_output_tokens', None) or 0
    return in_t, out_t

deadline = time.time() + 120
records = []
while time.time() < deadline:
    resp = fetch_traces(log_stream_id)
    records = getattr(resp, 'records', []) or []
    have_user_cost = any(extract_user_cost(r) is not None for r in records)
    have_metric_cost = any(extract_scorer_metric_cost(r, CUSTOM_METRIC_NAME)[0] == 'populated' for r in records)
    if records and have_user_cost and have_metric_cost:
        break
    time.sleep(5)

print(f'{"in/out":>10}  {"api cost":>14}  {"expected":>14}  {"metric_cost":>14}  {"judge_tok":>10}  prompt')
print('-' * 110)
total_user, total_metric = 0.0, 0.0
metric_cost_states = {'populated': 0, 'null': 0, 'absent': 0}
for r in records[:10]:
    in_t, out_t = extract_tokens(r)
    cost = extract_user_cost(r)
    state, metric_cost, judge_tokens = extract_scorer_metric_cost(r, CUSTOM_METRIC_NAME)
    metric_cost_states[state] += 1
    expected = (in_t * GPT_4O_MINI_IN_PER_M + out_t * GPT_4O_MINI_OUT_PER_M) / 1_000_000
    raw = getattr(r, 'input_', '') or ''
    prompt = (raw if isinstance(raw, str) else '')[:30]
    if isinstance(cost, (int, float)): total_user += cost
    if state == 'populated': total_metric += metric_cost
    metric_disp = f'${metric_cost:.6f}' if state == 'populated' else state
    judge_tok_disp = str(judge_tokens) if judge_tokens is not None else '-'
    print(
        f'{int(in_t):>4}/{int(out_t):>4}'
        f'  {(f"${cost:.6f}" if isinstance(cost,(int,float)) else "None"):>14}'
        f'  {f"${expected:.6f}":>14}'
        f'  {metric_disp:>14}'
        f'  {judge_tok_disp:>10}'
        f'  {prompt}'
    )

print(f'\nSum user cost                                = ${total_user:.6f}')
print(f'Sum {CUSTOM_METRIC_NAME}_metric_cost  = ${total_metric:.6f}'
      f'  (populated={metric_cost_states["populated"]},'
      f' null={metric_cost_states["null"]},'
      f' absent={metric_cost_states["absent"]})')

if metric_cost_states['populated'] == 0:
    diag = []
    if metric_cost_states['absent']:
        diag.append(f'  - {metric_cost_states["absent"]} record(s) absent: scorer Celery job not done yet — re-run after another minute.')
    if metric_cost_states['null']:
        diag.append(
            f'  - {metric_cost_states["null"]} record(s) null: judge ran but no cost was attributable.'
            f'\n    Likely causes: (a) {CUSTOM_JUDGE_MODEL} judge response missing token usage,'
            f'\n                   (b) override missing — re-check Step 3,'
            f'\n                   (c) comet (Celery Beat) not running.'
        )
    print('\n[WARN] No populated metric_cost yet. Diagnostic:')
    print('\n'.join(diag))

    in/out        api cost        expected     metric_cost   judge_tok  prompt
--------------------------------------------------------------------------------------------------------------
  36/  31       $0.000024       $0.000024            null           -  {"messages": [{"content": "You
  35/  33       $0.000025       $0.000025       $0.000000         831  {"messages": [{"content": "You
  34/  18       $0.000016       $0.000016       $0.000000         818  {"messages": [{"content": "You
  37/  21       $0.000018       $0.000018            null           -  {"messages": [{"content": "You
  33/  16       $0.000015       $0.000015            null           -  {"messages": [{"content": "You
  34/  34       $0.000025       $0.000025            null           -  {"messages": [{"content": "You

Sum user cost                                = $0.000123
Sum custom_LLM_boolean_metric_s16_metric_cost  = $0.000000  (populated=2, null=4, absent=0)


## Step 6b — §1.7: Verify `num_reasoning_tokens` via the **spans** search API

Important nuance: `num_reasoning_tokens` is recorded **per LLM span**, not on the trace. Trace-level metrics include roll-up aggregates of `num_input_tokens` / `num_output_tokens` / `num_total_tokens` (with `@sum`, `@min`, `@max`, `@average` suffixes), but reasoning tokens are *not* rolled up — so reading `record.metrics.additional_properties` from `query_traces_*` will return `None` even when the value is present on the underlying spans.

To verify the synthetic value Step 3b injected, we query spans directly with `query_spans_projects_project_id_spans_search_post` and read `span.metrics.additional_properties['num_reasoning_tokens']`.

In [10]:
from galileo.resources.api.trace import query_spans_projects_project_id_spans_search_post
from galileo.resources.models.log_records_sort_clause import LogRecordsSortClause

# Sort by created_at descending so we look at THIS run's spans, not stale ones from
# earlier runs that might pre-date the Step 3b monkey-patch.
sort_desc = LogRecordsSortClause(column_id='created_at', ascending=False)
span_resp = query_spans_projects_project_id_spans_search_post.sync(
    project_id=project_id, client=config.api_client,
    body=LogRecordsQueryRequest(log_stream_id=log_stream_id, limit=50, sort=sort_desc),
)
span_records = getattr(span_resp, 'records', []) or []
llm_spans = [s for s in span_records if getattr(s, 'type_', None) == 'llm']

# Verify against the count we just sent in Step 4 (one prompt = one llm span).
this_run = len(prompts)
target_spans = llm_spans[:this_run]

print(f'Total LLM spans returned: {len(llm_spans)}; verifying the most recent {len(target_spans)}.\n')
print(f'{"out_t":>6}  {"reasoning":>11}  {"expected":>11}  match')
print('-' * 50)

populated = matches = mismatches = 0
for s in target_spans:
    out_t = getattr(s.metrics, 'num_output_tokens', None) or 0
    ap = s.metrics.additional_properties or {}
    reasoning = ap.get('num_reasoning_tokens')
    expected = _synthesize_reasoning_tokens(int(out_t))
    match = reasoning is not None and int(reasoning) == expected
    if reasoning is not None:
        populated += 1
        if match:
            matches += 1
        else:
            mismatches += 1
    print(f'{int(out_t):>6}  {str(int(reasoning)) if reasoning is not None else "None":>11}  {expected:>11}  {"YES" if match else "no"}')

n = len(target_spans)
print(f'\nReasoning tokens (synthetic, native SDK path) on this run: populated={populated}/{n}; matches={matches}; mismatches={mismatches}')
if n > 0 and populated == n and matches == n and mismatches == 0:
    print(
        '\n  PROVEN (reasoning_tokens via native SDK):\n'
        '  Step 3b\'s monkey-patch on galileo.openai.extractors._parse_usage injected\n'
        '  reasoning_tokens = (output_tokens // 2) + 7 into the parsed usage dict.\n'
        '  galileo/openai/__init__.py:248 then wrote span.metrics.num_reasoning_tokens via\n'
        '  Pydantic extra="allow" on LlmMetrics. The value surfaces verbatim on each LLM span.\n'
        '  Reminder: it does NOT roll up to the trace, so trace-level queries return None.'
    )

Total LLM spans returned: 6; verifying the most recent 6.

 out_t    reasoning     expected  match
--------------------------------------------------
    31           22           22  YES
    33           23           23  YES
    18           16           16  YES
    21           17           17  YES
    16           15           15  YES
    34           24           24  YES

Reasoning tokens (synthetic, native SDK path) on this run: populated=6/6; matches=6; mismatches=0

  PROVEN (reasoning_tokens via native SDK):
  Step 3b's monkey-patch on galileo.openai.extractors._parse_usage injected
  reasoning_tokens = (output_tokens // 2) + 7 into the parsed usage dict.
  galileo/openai/__init__.py:248 then wrote span.metrics.num_reasoning_tokens via
  Pydantic extra="allow" on LlmMetrics. The value surfaces verbatim on each LLM span.
  Reminder: it does NOT roll up to the trace, so trace-level queries return None.


## Step 7: Job progress

If metric_cost stays 0, check job state. Look for `log_stream_scorer` jobs targeting this log stream's `run_id`.

In [11]:
jobs = config.api_client.request(
    method=RequestMethod.GET,
    path=f'/projects/{project_id}/runs/{log_stream_id}/jobs',
) or []

print(f'Total jobs for this log stream: {len(jobs)}\n')
print(f'{"job_name":<28} {"status":<14} {"progress":>10}  message')
print('-' * 80)
for j in jobs[:15]:
    name = (j.get('job_name') or '')[:28]
    status = (j.get('status') or '')[:14]
    sc, st = j.get('steps_completed'), j.get('steps_total')
    pct = f'{(sc/st*100):.0f}%' if (sc is not None and st) else '-'
    msg = (j.get('progress_message') or '')[:40]
    print(f'{name:<28} {status:<14} {pct:>10}  {msg}')

Total jobs for this log stream: 4

job_name                     status           progress  message
--------------------------------------------------------------------------------
log_stream_scorer            in_progress             -  
log_stream_scorer            completed               -  
log_stream_scorer            completed               -  
log_stream_scorer            completed               -  


## Step 8: Dashboard surfaces

In [12]:
print('=== UI surfaces ===')
print(f'Log stream       : {log_stream_url}')
print(f'Trends tab       : {trends_url}')
print(f'Settings/pricing : {urljoin(console_base, "settings/model-pricing")}')

=== UI surfaces ===
Log stream       : https://console-bin-citizens.gcp-dev.galileo.ai/project/af0c35bd-a7b1-4bef-ad32-120a20cec15d/log-streams/8f80737e-68aa-494d-a910-7c1976180b74
Trends tab       : https://console-bin-citizens.gcp-dev.galileo.ai/project/af0c35bd-a7b1-4bef-ad32-120a20cec15d/log-streams/8f80737e-68aa-494d-a910-7c1976180b74?tab=trends
Settings/pricing : https://console-bin-citizens.gcp-dev.galileo.ai/settings/model-pricing


## Step 9: Cleanup

In [13]:
# delete_project(name=PROJECT_NAME)
print(f'Project kept for inspection: {PROJECT_NAME}')
print(f'  Log stream : {log_stream_url}')

Project kept for inspection: GalileoEval_S16_CostCustomJudge
  Log stream : https://console-bin-citizens.gcp-dev.galileo.ai/project/af0c35bd-a7b1-4bef-ad32-120a20cec15d/log-streams/8f80737e-68aa-494d-a910-7c1976180b74
